# Réemploi des plaques gravées entre éditeurs

Une ligne = un jeu de plaques (identifié par le graveur), un point = une édition qui
l'utilise, positionnée par année. Trace deux choses distinctes : les segments de
réemploi (des éditions successives du même jeu de plaques, chez le même éditeur ou
transmises à un autre) et les flèches de copie (un jeu de plaques copie l'iconographie
d'un autre, résolu depuis la colonne texte libre "copies de cette édition").

La logique de lecture/regroupement/résolution vit dans `reemploi_plaques_utils.py`,
partagée avec [`reemploi_plaques_editeur.py`](reemploi_plaques_editeur.py) — l'atelier
web qui permet de corriger l'ordre des lignes et les liens directement depuis la
visualisation (au lieu d'éditer le tableau source à la main). Ce notebook lit les mêmes
corrections (`retours_celine/corrections_reemploi_plaques.json`, si présent) pour que
l'export statique reste cohérent avec ce qui a été corrigé dans l'atelier.

In [1]:
import os

import reemploi_plaques_utils as u

RACINE = os.path.abspath("../..")
DOSSIER_VIZ = os.path.join(RACINE, "resultats", "Datavis")
os.makedirs(DOSSIER_VIZ, exist_ok=True)
CHEMIN_SORTIE = os.path.join(DOSSIER_VIZ, "reemploi_plaques.html")
CHEMIN_CORPUS = os.path.join(RACINE, "retours_celine", "BNU_corpus.ods")
CHEMIN_CORRECTIONS = os.path.join(RACINE, "retours_celine", "corrections_reemploi_plaques.json")


## Construction (lecture, regroupement, résolution des copies, mise en page)

Une seule fonction : voir `reemploi_plaques_utils.construire_tout()` pour le détail de
chaque étape (elle affiche les mêmes statistiques que l'ancienne version cellule par
cellule — nombre d'éditions retenues, jeux de plaques, segments par type, flèches de
copie résolues/non résolues).

In [2]:
resultat = u.construire_tout(CHEMIN_CORPUS, chemin_corrections=CHEMIN_CORRECTIONS)

plaques = resultat["plaques"]
points = resultat["points"]
elements_svg = resultat["elements_svg"]
non_resolus_copies = resultat["non_resolus_copies"]

if resultat["corrections"].get("liens") or resultat["corrections"].get("ordre"):
    print(f"\n{len(resultat['corrections'].get('liens', []))} correction(s) manuelle(s) appliquée(s) "
          f"depuis {CHEMIN_CORRECTIONS}")

for p in plaques[:10]:
    chaine = " → ".join(f"{e['publisher']} ({e['annee']})" for e in p["editions"])
    print(f"  {p['graveur']:30s} {chaine}")


4 lignes fusionnées (tomes d'une même édition regroupés)
99 éditions avec un graveur identifiable, sur 107 au total
7 éditions écartées (graveur composite, plusieurs personnes citées)
45 jeux de plaques distincts
45 jeux de plaques au total — 23 réemployés (≥2 éditions), affichés dans la frise
33 transmissions à un autre éditeur, 13 réimpressions par le même éditeur, 8 cas incertains
19 flèches de copie affichées 
11 mentions non résolues ou non affichables

3 correction(s) manuelle(s) appliquée(s) depuis /mnt/c/Users/a.saidi/OneDrive - BNU/Bureau/working_dir/retours_celine/corrections_reemploi_plaques.json
  Solis, Virgil                  Sigmund Fereyabend (1563) → Georg Raben (1564) → G. Corvinum, Sigmund Feyerabent et W. Galli (1569) → Georges Corvin et héritiers  Vuigandi Galli (1571) → Raben, Feyerabend et Han (1571) → Johann Feyerabendt (in verlegung Sigmund Feyerabendts) (1581) → Pedro Bellero (1595) → Johann Saurn (in Verlegung Francisci Nicolai Rothen) (1609) → G. Leestens (1

### Mentions de copie non résolues

Cas non affichables automatiquement (attribution ambiguë, graveur cité introuvable,
pas d'édition antérieure affichée...) — corrigeables un par un dans l'atelier web
(`reemploi_plaques_editeur.py`) via un ajout manuel, si la lecture du tableau source
permet de lever l'ambiguïté.

In [3]:
for e, texte, cible, raison in non_resolus_copies[:15]:
    print(f"  ✗ {e['graveur']:20s} {e['annee']} — {texte!r} — {cible!r} ({raison})")
if len(non_resolus_copies) > 15:
    print(f"  … et {len(non_resolus_copies) - 15} de plus")


  ✗ Anonyme1526          1526 — 'copie QUOI\xa0?' — '(source)' (édition source non affichée (jeu de plaques jamais réemployé))
  ✗ Anonyme1527          1527 — 'copie Leroy II, Guillaume ou 1497' — '(toute la mention)' (attribution ambiguë (ou/possibilité multiple))
  ✗ Anonyme1532          1532 — 'copie Leroy II, Guillaume ou Anonyme1497 ou Anonyme1527' — '(toute la mention)' (attribution ambiguë (ou/possibilité multiple))
  ✗ Eskrich, Pierre      1556 — 'copie Bernard Salomon' — 'Salomon, Bernard' (pas d'édition antérieure affichée pour ce graveur)
  ✗ Passe, Crispin de    1602 — 'copie van der Borcht\xa0? Salomon\xa0?' — '(toute la mention)' (attribution ambiguë (ou/possibilité multiple))
  ✗ Philippe, Pierre     1662 — 'copie Clein, Francisco (inv.) et Savery, Salomon (sculp.)' — 'Clein, Francisco' (aucune correspondance)
  ✗ Weyen, Laurent       1669 — 'copie Clein, Francisco (inv.) et Savery, Salomon (sculp.) ou Philippe, Pierre (inversé)' — '(toute la mention)' (attribution ambig

### Export HTML

In [4]:
LIGNES_TABLEAU = u.construire_lignes_tableau(plaques)

html_final = u.generer_html(
    largeur=resultat["largeur"], hauteur=resultat["hauteur"],
    frise_svg="\n".join(elements_svg), lignes_tableau=LIGNES_TABLEAU,
    points=points, libelles_technique=u.LIBELLES_TECHNIQUE,
    editable=False,
)

with open(CHEMIN_SORTIE, "w", encoding="utf-8") as f:
    f.write(html_final)

print("Frise écrite dans", CHEMIN_SORTIE)


Frise écrite dans /mnt/c/Users/a.saidi/OneDrive - BNU/Bureau/working_dir/resultats/Datavis/reemploi_plaques.html
